In [ ]:
#@title 🎧 Download Narration Audio & Play Introduction
import os as _os
if not _os.path.exists("/content/narration"):
    !pip install -q gdown
    import gdown
    gdown.download(id="17YTdmFwvXA58S8IhKAffVY_OC1IcIi2p", output="/content/narration.zip", quiet=False)
    !unzip -q /content/narration.zip -d /content/narration
    !rm /content/narration.zip
    print(f"Loaded {len(_os.listdir('/content/narration'))} narration segments")
else:
    print("Narration audio already loaded.")

from IPython.display import Audio, display
display(Audio("/content/narration/01_00_intro.mp3"))


# Pipeline Parallelism: The Pipeline Bubble -- Vizuara

## What You Will Learn

In this notebook, you will:

1. **Understand the naive pipeline schedule** and why it creates massive idle time
2. **Simulate a multi-GPU pipeline** using Python (no actual multi-GPU hardware needed)
3. **Measure the bubble fraction** analytically and verify it with simulation
4. **Visualize pipeline timelines** to see exactly where GPUs sit idle
5. **Explore how micro-batching reduces the bubble** as a preview of GPipe

**Estimated time**: 40-60 minutes

**Prerequisites**: Basic Python, familiarity with neural network training (forward/backward pass)

In [ ]:
#@title 🎧 Code Walkthrough: Setup
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_01_setup.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Setup: Run this cell first!
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from dataclasses import dataclass, field
from typing import List, Tuple, Optional
import time

%matplotlib inline

# Reproducibility
np.random.seed(42)

print("Setup complete!")
print("This notebook simulates pipeline parallelism -- no GPU required.")
print("We model pipeline schedules and measure bubble fractions.")

In [ ]:
#@title 🎧 Listen: Assembly Line Analogy
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_02_assembly_line_analogy.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 1. The Assembly Line Analogy

Pipeline parallelism splits the **layers** of a neural network across GPUs. GPU 0 handles the first few layers, GPU 1 the next, and so on. Data flows through the GPUs in sequence, like cars on an assembly line.

The key question is: **how do we schedule work so that all GPUs stay busy?**

Let us start with the simplest (and worst) approach -- the **naive pipeline** -- and measure exactly how bad it is.

In [ ]:
#@title 🎧 Code Walkthrough: Pipeline Stage Definition
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_03_pipeline_stage_definition.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# A "stage" represents one group of layers on one GPU.
# We model each stage by its forward and backward compute times.

@dataclass
class PipelineStage:
    """One pipeline stage (group of layers on one GPU)."""
    stage_id: int
    forward_time: float   # milliseconds for one forward pass
    backward_time: float  # milliseconds for one backward pass

@dataclass
class ScheduleEvent:
    """One scheduled operation on a GPU."""
    stage_id: int
    microbatch_id: int
    is_forward: bool        # True = forward, False = backward
    start_time: float
    end_time: float

def create_uniform_pipeline(num_stages: int, forward_time: float = 100.0) -> List[PipelineStage]:
    """Create a pipeline with uniform stage times.
    Backward pass is ~2x forward (gradient w.r.t. activations + weights)."""
    return [
        PipelineStage(
            stage_id=i,
            forward_time=forward_time,
            backward_time=forward_time * 2.0
        )
        for i in range(num_stages)
    ]

# Create a 4-stage pipeline
stages = create_uniform_pipeline(num_stages=4, forward_time=100.0)

for s in stages:
    print(f"Stage {s.stage_id}: forward={s.forward_time:.0f}ms, backward={s.backward_time:.0f}ms")

print(f"\nTotal layers spread across {len(stages)} GPUs.")
print(f"Each forward pass: {stages[0].forward_time:.0f}ms per stage")
print(f"Each backward pass: {stages[0].backward_time:.0f}ms per stage")

In [ ]:
#@title 🎧 Listen: Naive Schedule Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_04_naive_schedule_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 2. The Naive Pipeline Schedule

The naive schedule processes **one micro-batch at a time** through the entire pipeline:

1. Forward pass flows from stage 0 to stage P-1
2. Backward pass flows from stage P-1 back to stage 0
3. Only then does the next micro-batch begin

This means at any given moment, **only one GPU is active**. All others are idle.

In [ ]:
#@title 🎧 Code Walkthrough: Naive Schedule Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_05_naive_schedule_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def schedule_naive(stages: List[PipelineStage], num_microbatches: int) -> List[ScheduleEvent]:
    """Naive pipeline: one micro-batch at a time, fully sequential."""
    events = []
    P = len(stages)
    current_time = 0.0

    for mb in range(num_microbatches):
        # Forward pass: stage 0 -> stage P-1
        for s in range(P):
            duration = stages[s].forward_time
            events.append(ScheduleEvent(
                stage_id=s,
                microbatch_id=mb,
                is_forward=True,
                start_time=current_time,
                end_time=current_time + duration
            ))
            current_time += duration

        # Backward pass: stage P-1 -> stage 0
        for s in range(P - 1, -1, -1):
            duration = stages[s].backward_time
            events.append(ScheduleEvent(
                stage_id=s,
                microbatch_id=mb,
                is_forward=False,
                start_time=current_time,
                end_time=current_time + duration
            ))
            current_time += duration

    return events

# Schedule 1 micro-batch through 4 stages
events_naive = schedule_naive(stages, num_microbatches=1)

print("Naive pipeline schedule (1 micro-batch, 4 stages):")
print(f"{'Stage':>6} {'Type':>8} {'MB':>4} {'Start':>8} {'End':>8} {'Duration':>10}")
print("-" * 52)
for e in events_naive:
    op = "FWD" if e.is_forward else "BWD"
    print(f"{e.stage_id:>6} {op:>8} {e.microbatch_id:>4} {e.start_time:>8.0f} {e.end_time:>8.0f} {e.end_time - e.start_time:>10.0f}")

total_time = max(e.end_time for e in events_naive)
print(f"\nTotal pipeline time: {total_time:.0f} ms")

In [ ]:
#@title 🎧 Listen: Bubble Measurement Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_06_bubble_measurement_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 3. Measuring the Pipeline Bubble

The **pipeline bubble** is the fraction of total GPU-time that is wasted (idle). For the naive schedule with $P$ stages and 1 micro-batch:

$$\text{Bubble fraction} = \frac{P - 1}{P}$$

Let us verify this with our simulation.

In [ ]:
#@title 🎧 Code Walkthrough: Compute Bubble Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_07_compute_bubble_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def compute_bubble_fraction(events: List[ScheduleEvent], num_stages: int) -> dict:
    """Compute bubble fraction from schedule events."""
    total_time = max(e.end_time for e in events)

    # Total GPU-time available = num_stages * total_time
    total_gpu_time = num_stages * total_time

    # Compute busy time per stage
    busy_per_stage = {s: 0.0 for s in range(num_stages)}
    for e in events:
        busy_per_stage[e.stage_id] += (e.end_time - e.start_time)

    total_busy = sum(busy_per_stage.values())
    total_idle = total_gpu_time - total_busy
    bubble_fraction = total_idle / total_gpu_time

    return {
        "total_time": total_time,
        "total_gpu_time": total_gpu_time,
        "total_busy": total_busy,
        "total_idle": total_idle,
        "bubble_fraction": bubble_fraction,
        "busy_per_stage": busy_per_stage
    }

# Analyze the naive schedule
P = 4
stats = compute_bubble_fraction(events_naive, P)

print(f"Pipeline Analysis (Naive, P={P}, M=1):")
print(f"  Total wall-clock time: {stats['total_time']:.0f} ms")
print(f"  Total GPU-time (all GPUs): {stats['total_gpu_time']:.0f} ms")
print(f"  Total busy time: {stats['total_busy']:.0f} ms")
print(f"  Total idle time: {stats['total_idle']:.0f} ms")
print(f"  Bubble fraction: {stats['bubble_fraction']:.2%}")
print()

# Compare with the formula
theoretical_bubble = (P - 1) / P
print(f"Theoretical bubble: (P-1)/P = ({P-1}/{P}) = {theoretical_bubble:.2%}")
print(f"Simulated bubble:   {stats['bubble_fraction']:.2%}")
print(f"Match: {abs(stats['bubble_fraction'] - theoretical_bubble) < 1e-6}")

In [ ]:
#@title 🎧 Listen: Visualization Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_08_visualization_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 4. Visualizing the Pipeline Timeline

A timeline visualization makes the waste crystal clear. Each row is a GPU, and we color forward passes blue, backward passes red, and idle time gray.

In [ ]:
#@title 🎧 What to Look For: Plot Timeline Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_09_plot_timeline_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def plot_pipeline_timeline(events: List[ScheduleEvent], num_stages: int, title: str,
                           figsize=(16, 5)):
    """Visualize a pipeline schedule as a Gantt chart."""
    fig, ax = plt.subplots(figsize=figsize)
    total_time = max(e.end_time for e in events)

    # Color scheme
    fwd_color = '#4A90D9'   # blue for forward
    bwd_color = '#E8634F'   # red for backward
    idle_color = '#E8E8E8'  # light gray for idle

    # Draw idle background
    for s in range(num_stages):
        ax.barh(s, total_time, left=0, height=0.7, color=idle_color, edgecolor='white', linewidth=0.5)

    # Draw active blocks
    for e in events:
        color = fwd_color if e.is_forward else bwd_color
        duration = e.end_time - e.start_time
        ax.barh(e.stage_id, duration, left=e.start_time, height=0.7,
                color=color, edgecolor='white', linewidth=1.0)

        # Label
        label = f"F{e.microbatch_id}" if e.is_forward else f"B{e.microbatch_id}"
        ax.text(e.start_time + duration / 2, e.stage_id, label,
                ha='center', va='center', fontsize=8, fontweight='bold', color='white')

    ax.set_yticks(range(num_stages))
    ax.set_yticklabels([f'GPU {i}' for i in range(num_stages)])
    ax.set_xlabel('Time (ms)', fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.invert_yaxis()

    # Legend
    legend_elements = [
        mpatches.Patch(facecolor=fwd_color, label='Forward'),
        mpatches.Patch(facecolor=bwd_color, label='Backward'),
        mpatches.Patch(facecolor=idle_color, label='Idle (bubble)'),
    ]
    ax.legend(handles=legend_elements, loc='upper right', fontsize=10)

    # Compute and annotate bubble
    stats = compute_bubble_fraction(events, num_stages)
    ax.text(0.02, 0.02, f"Bubble: {stats['bubble_fraction']:.1%}",
            transform=ax.transAxes, fontsize=12, fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

    plt.tight_layout()
    plt.show()

# Visualize the naive schedule
plot_pipeline_timeline(events_naive, P, "Naive Pipeline (P=4, M=1): 75% Bubble")

In [ ]:
#@title 🎧 Listen: Timeline Explanation
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_10_timeline_explanation.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


The visualization makes it abundantly clear: each GPU is active for only 2 time slots out of 8 (one forward, one backward). The rest is idle time -- the **pipeline bubble**.

In [ ]:
#@title 🎧 Listen: Bubble Scales Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_11_bubble_scales_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 5. How the Bubble Scales with Pipeline Depth

The bubble fraction for the naive schedule is $(P-1)/P$. Let us see how it grows as we add more stages.

In [ ]:
#@title 🎧 What to Look For: Bubble Scales Plot
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_12_bubble_scales_plot.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Bubble fraction vs number of pipeline stages (naive schedule)
stage_counts = list(range(2, 17))
bubble_fractions_naive = [(p - 1) / p for p in stage_counts]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(stage_counts, [b * 100 for b in bubble_fractions_naive], 'o-',
        color='#E8634F', markersize=8, linewidth=2, label='Naive pipeline')

# Annotate key points
for p in [2, 4, 8, 16]:
    b = (p - 1) / p * 100
    ax.annotate(f'{b:.1f}%', (p, b), textcoords="offset points",
                xytext=(10, 5), fontsize=10)

ax.set_xlabel('Number of Pipeline Stages (P)', fontsize=12)
ax.set_ylabel('Bubble Fraction (%)', fontsize=12)
ax.set_title('Naive Pipeline: Bubble Grows with Depth', fontsize=14, fontweight='bold')
ax.set_ylim(0, 100)
ax.grid(True, alpha=0.3)
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

print("Key insight: With the naive schedule, more stages means MORE wasted time.")
print("At P=8, 87.5% of GPU time is idle. This is clearly unacceptable.")

In [ ]:
#@title 🎧 Listen: Microbatching Fix Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_13_microbatching_fix_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 6. The Fix: Multiple Micro-Batches

The solution to the bubble problem is to send **multiple micro-batches** through the pipeline. While GPU 3 works on micro-batch 1, GPU 2 can start micro-batch 2, and so on.

Let us implement a basic micro-batch pipelining schedule (a simplified version of GPipe) and see how the bubble shrinks.

In [ ]:
#@title 🎧 Code Walkthrough: Gpipe Simple Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_14_gpipe_simple_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def schedule_gpipe_simple(stages: List[PipelineStage], num_microbatches: int) -> List[ScheduleEvent]:
    """Simplified GPipe: all forwards first, then all backwards.
    Micro-batches are pipelined -- the next micro-batch starts on a stage
    as soon as the previous one finishes."""
    events = []
    P = len(stages)
    M = num_microbatches

    # Track when each stage becomes free
    stage_free_at = [0.0] * P

    # Phase 1: All forward passes
    for mb in range(M):
        for s in range(P):
            # Must wait for: (a) this stage to be free, (b) previous stage to finish this MB
            earliest = stage_free_at[s]
            if s > 0:
                # Find when previous stage finished forward for this MB
                prev_fwd = [e for e in events if e.stage_id == s - 1
                           and e.microbatch_id == mb and e.is_forward]
                if prev_fwd:
                    earliest = max(earliest, prev_fwd[0].end_time)

            start = earliest
            duration = stages[s].forward_time
            end = start + duration

            events.append(ScheduleEvent(
                stage_id=s, microbatch_id=mb,
                is_forward=True, start_time=start, end_time=end
            ))
            stage_free_at[s] = end

    # Phase 2: All backward passes (in reverse micro-batch order)
    for mb in range(M - 1, -1, -1):
        for s in range(P - 1, -1, -1):
            earliest = stage_free_at[s]
            if s < P - 1:
                prev_bwd = [e for e in events if e.stage_id == s + 1
                           and e.microbatch_id == mb and not e.is_forward]
                if prev_bwd:
                    earliest = max(earliest, prev_bwd[0].end_time)

            start = earliest
            duration = stages[s].backward_time
            end = start + duration

            events.append(ScheduleEvent(
                stage_id=s, microbatch_id=mb,
                is_forward=False, start_time=start, end_time=end
            ))
            stage_free_at[s] = end

    return events

# Compare naive (M=1) vs pipelined (M=4, M=8)
P = 4
stages = create_uniform_pipeline(P, forward_time=100.0)

for M in [1, 4, 8, 16]:
    if M == 1:
        events = schedule_naive(stages, M)
    else:
        events = schedule_gpipe_simple(stages, M)
    stats = compute_bubble_fraction(events, P)
    theoretical = (P - 1) / (P - 1 + M)
    print(f"M={M:>2}: Bubble={stats['bubble_fraction']:.1%}  "
          f"(theoretical: {theoretical:.1%})  "
          f"Total time: {stats['total_time']:.0f}ms")

In [ ]:
#@title 🎧 What to Look For: Gpipe M4 Viz
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_15_gpipe_m4_viz.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Visualize M=4 schedule
events_m4 = schedule_gpipe_simple(stages, num_microbatches=4)
plot_pipeline_timeline(events_m4, P, "Pipelined (P=4, M=4): Bubble Reduced")

In [ ]:
#@title 🎧 What to Look For: Gpipe M8 Viz
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_16_gpipe_m8_viz.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Visualize M=8 schedule
events_m8 = schedule_gpipe_simple(stages, num_microbatches=8)
plot_pipeline_timeline(events_m8, P, "Pipelined (P=4, M=8): Bubble Shrinks Further",
                       figsize=(20, 5))

In [ ]:
#@title 🎧 Listen: Bubble Vs M Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_17_bubble_vs_m_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 7. Bubble Fraction vs. Number of Micro-Batches

The bubble fraction with $M$ micro-batches is:

$$\text{Bubble} = \frac{P - 1}{P - 1 + M}$$

As $M \to \infty$, the bubble goes to zero. Let us plot this relationship.

In [ ]:
#@title 🎧 What to Look For: Bubble Vs M Plot
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_18_bubble_vs_m_plot.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

M_values = np.arange(1, 65)

for P in [2, 4, 8, 16]:
    bubble = (P - 1) / (P - 1 + M_values) * 100
    ax.plot(M_values, bubble, linewidth=2, label=f'P={P} stages')

# Mark the "practical" threshold
ax.axhline(y=10, color='gray', linestyle='--', alpha=0.5)
ax.text(55, 11, '10% threshold', fontsize=10, color='gray')

ax.set_xlabel('Number of Micro-Batches (M)', fontsize=12)
ax.set_ylabel('Bubble Fraction (%)', fontsize=12)
ax.set_title('Bubble Fraction vs. Micro-Batches for Different Pipeline Depths',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 80)
plt.tight_layout()
plt.show()

print("Rule of thumb: M >= 4*P keeps the bubble below ~10%.")
for P in [4, 8, 16]:
    M_needed = 4 * P
    bubble = (P - 1) / (P - 1 + M_needed) * 100
    print(f"  P={P}: need M>={M_needed}, bubble={bubble:.1f}%")

In [ ]:
#@title 🎧 Listen: Communication Cost Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_19_communication_cost_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 8. Communication Cost: How Big Are Activation Transfers?

Between pipeline stages, the only data transferred is the **activation tensor** at the boundary. Let us calculate how much data this is.

In [ ]:
#@title 🎧 Code Walkthrough: Activation Transfer Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_20_activation_transfer_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def activation_transfer_size(batch_size: int, seq_len: int, hidden_dim: int,
                              dtype_bytes: int = 2) -> float:
    """Compute activation tensor size in MB.

    Args:
        batch_size: Micro-batch size
        seq_len: Sequence length
        hidden_dim: Hidden dimension
        dtype_bytes: 2 for FP16/BF16, 4 for FP32
    Returns:
        Size in MB
    """
    return batch_size * seq_len * hidden_dim * dtype_bytes / (1024 ** 2)

print("Activation transfer sizes between pipeline stages:")
print(f"{'Model':>20} {'b':>4} {'s':>6} {'h':>6} {'FP16 (MB)':>10} {'FP32 (MB)':>10}")
print("-" * 62)

configs = [
    ("GPT-2 (small)",   1, 1024, 768),
    ("GPT-2 (large)",   1, 1024, 1280),
    ("LLaMA 7B",        1, 4096, 4096),
    ("LLaMA 70B",       1, 4096, 8192),
    ("GPT-3 175B",      1, 2048, 12288),
]

for name, b, s, h in configs:
    fp16 = activation_transfer_size(b, s, h, dtype_bytes=2)
    fp32 = activation_transfer_size(b, s, h, dtype_bytes=4)
    print(f"{name:>20} {b:>4} {s:>6} {h:>6} {fp16:>10.1f} {fp32:>10.1f}")

print("\nKey insight: Pipeline communication is just one activation tensor (~10-100 MB).")
print("Compare this to data parallel allreduce which communicates ALL gradients.")

In [ ]:
#@title 🎧 Before You Start: Todo Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_21_todo_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 9. Your Turn

### Exercise 1: Compute Bubble Fractions

Given different configurations, compute the bubble fraction analytically and verify with the simulator.

In [ ]:
#@title 🎧 Before You Start: Todo Bubble Fraction
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_22_todo_1_bubble_fraction.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# TODO: For each configuration, compute the theoretical bubble fraction
# and then verify with the simulator.

configs = [
    {"P": 4, "M": 4},
    {"P": 4, "M": 16},
    {"P": 8, "M": 8},
    {"P": 8, "M": 32},
    {"P": 16, "M": 64},
]

for cfg in configs:
    P, M = cfg["P"], cfg["M"]

    # TODO: Compute theoretical bubble fraction using the formula
    # theoretical_bubble = ???
    theoretical_bubble = None  # REPLACE THIS

    # Verify with simulator
    test_stages = create_uniform_pipeline(P, forward_time=50.0)
    test_events = schedule_gpipe_simple(test_stages, M)
    test_stats = compute_bubble_fraction(test_events, P)

    print(f"P={P:>2}, M={M:>2}: theoretical={theoretical_bubble}, "
          f"simulated={test_stats['bubble_fraction']:.3f}")

In [ ]:
#@title 🎧 Before You Start: Todo Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_23_todo_2_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### Exercise 2: Non-Uniform Stage Times

In practice, the first and last stages often take different amounts of time (e.g., the embedding layer is cheaper, the final projection + loss is heavier). Simulate a non-uniform pipeline and see how it affects the bubble.

In [ ]:
#@title 🎧 Before You Start: Todo Non Uniform
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_24_todo_2_non_uniform.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# TODO: Create a non-uniform 4-stage pipeline where:
# - Stage 0 (embedding): forward=50ms, backward=50ms
# - Stage 1 (transformer layers): forward=100ms, backward=200ms
# - Stage 2 (transformer layers): forward=100ms, backward=200ms
# - Stage 3 (output + loss): forward=120ms, backward=180ms

non_uniform_stages = [
    # TODO: Create PipelineStage objects with the times above
    # PipelineStage(stage_id=0, forward_time=???, backward_time=???),
    # ...
]

# After creating the stages, schedule and visualize:
# events_nonuniform = schedule_gpipe_simple(non_uniform_stages, num_microbatches=8)
# stats_nonuniform = compute_bubble_fraction(events_nonuniform, 4)
# plot_pipeline_timeline(events_nonuniform, 4, "Non-Uniform Pipeline (P=4, M=8)")
# print(f"Bubble fraction: {stats_nonuniform['bubble_fraction']:.1%}")

# QUESTION: Is the bubble larger or smaller than the uniform case? Why?

In [ ]:
#@title 🎧 Before You Start: Todo Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_25_todo_3_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### Exercise 3: Throughput Analysis

Compute the **effective throughput** (micro-batches per second) for different configurations. How does throughput scale with $M$?

In [ ]:
#@title 🎧 Before You Start: Todo Throughput
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_26_todo_3_throughput.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# TODO: For P=4 with forward_time=100ms, compute throughput for M=1,2,4,8,16,32,64
#
# Throughput = M / total_time (micro-batches per second)
# Note: total_time is in milliseconds, so multiply by 1000 for per-second

P = 4
stages = create_uniform_pipeline(P, forward_time=100.0)

M_values = [1, 2, 4, 8, 16, 32, 64]
throughputs = []

for M in M_values:
    events = schedule_gpipe_simple(stages, M)
    total_time = max(e.end_time for e in events)

    # TODO: Compute throughput in micro-batches per second
    throughput = None  # REPLACE THIS
    throughputs.append(throughput)
    print(f"M={M:>3}: total_time={total_time:>8.0f}ms, throughput={throughput} MB/s")

# TODO: Plot throughput vs M
# What do you observe? Does throughput increase linearly with M?

In [ ]:
#@title 🎧 Transition: Summary
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_27_summary.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 10. Summary

In this notebook, you learned:

1. **Pipeline parallelism** splits model layers across GPUs, with data flowing through stages sequentially.

2. The **naive pipeline** processes one micro-batch at a time, resulting in a bubble fraction of $(P-1)/P$ -- completely unacceptable for $P > 2$.

3. **Micro-batching** reduces the bubble to $(P-1)/(P-1+M)$, which approaches zero as $M$ grows.

4. The **communication cost** between stages is a single activation tensor, which is much smaller than the allreduce in data parallelism.

5. A practical rule of thumb: use $M \geq 4P$ to keep the bubble below 10%.

**Next notebook**: We will implement the GPipe and 1F1B schedules, and discover that 1F1B achieves the same bubble as GPipe while using dramatically less memory.

In [ ]:
#@title 🎧 Wrap-Up: Closing Summary Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_28_closing_summary_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
print("=" * 60)
print("SUMMARY: Pipeline Bubble")
print("=" * 60)
print()
print("Key formulas:")
print("  Naive bubble:  (P-1)/P")
print("  With M micro-batches: (P-1)/(P-1+M)")
print()
print("Example (P=4):")
print(f"  M=1:  bubble = {(3/4)*100:.1f}%")
print(f"  M=4:  bubble = {(3/7)*100:.1f}%")
print(f"  M=16: bubble = {(3/19)*100:.1f}%")
print(f"  M=64: bubble = {(3/67)*100:.1f}%")
print()
print("Next: GPipe and 1F1B schedules!")